In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import ee
import geemap
import pandas as pd
import os
import json
import geopandas as gpd
import geemap
import time

# Initialize input your project name to run this code
ee.Authenticate()
ee.Initialize(project='')

In [1]:
import ee
import time


# =========================================================
# 1. SETUP ASSETS AND HYBRID BOUNDARIES
# =========================================================

# Population Layers
org_childpop = ee.ImageCollection("projects/unicef-ccri/assets/misc_population/WorldPop_Con_T_U18")
#childpop = org_childpop.mosaic()
childpop = org_childpop.mosaic().setDefaultProjection(
    org_childpop.first().projection()
)
pop_res = org_childpop.first().projection().nominalScale()


totalpop = ee.Image("projects/unicef-ccri/assets/misc_population/worldpop_1km")
totalpop_res = totalpop.projection().nominalScale()

# Projection Reference
referenceImage = ee.Image("projects/unicef-ccri/assets/heatwaves/heatwave_frequency_2014_2023_avg")
targetScale = referenceImage.projection().nominalScale()
targetCRS = referenceImage.projection()

# Load Boundaries
detailedBoundaries = ee.FeatureCollection('projects/unicef-ccri/assets/global_boundary/adm0')
# Global geometry (helper for mean reductions)
global_geometry = ee.Geometry.Polygon(
    [[[-180, 90], [-180, -90], [180, -90], [180, 90]]], None, False
)
simple_fc = ee.FeatureCollection("projects/unicef-ccri/assets/misc_boundaries/adm0_simple")
global_geom = simple_fc.geometry()

# Function to create hybrid boundaries (simplify complex ones)
def make_hybrid_boundary(feature):
    geometry = feature.geometry()
    area = geometry.area()

    # Determine simplified geometry based on area size
    simplifiedGeometry = ee.Algorithms.If(
        area.lt(5e8),
        geometry,
        ee.Algorithms.If(
            area.lt(3e11),
            geometry.simplify(100),
            geometry.simplify(10000)
        )
    )
    # Transform to target CRS
    finalGeometry = ee.Geometry(simplifiedGeometry).transform(targetCRS)
    return feature.setGeometry(finalGeometry)

hybridBoundaries = detailedBoundaries.map(make_hybrid_boundary)

# =========================================================
# 2. HAZARD & TOPIC DEFINITIONS
# =========================================================

hazards = [
    {"id": "projects/unicef-ccri/assets/hazards/river_flood_r100", "threshold": 0.01, "name": "river_flood_100yr_jrc_2024", "type": "Collection"},
    {"id": "projects/unicef-ccri/assets/hazards/coastal_flood_r100", "threshold": 0, "name": "coastal_flood_100yr_jrc_2024", "type": "Collection"},
    {"id": "projects/unicef-ccri/assets/hazards/storm_giri_rp100", "threshold": 17.5, "name": "tropical_storm_100yr_giri_2024", "type": "Collection"},
    {"id": "projects/unicef-ccri/assets/hazards/ASI_return_level_100yr", "threshold": 30, "name": "agricultural_drought_fao_1984-2023", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/droughts/drought_spei_copernicus_1940_2024", "threshold": -1.5, "name": "drought_spei_copernicus_1940-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/droughts/drought_spi_copernicus_1940_2024", "threshold": -1.5, "name": "drought_spi_copernicus_1940-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/heatwave_frequency_return_level_100yr", "threshold": 16.02, "name": "heatwave_frequency_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/heatwave_duration_return_level_100yr", "threshold": 94.01, "name": "heatwave_duration_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/heatwave_severity_return_level_100yr", "threshold": 3.66, "name": "heatwave_severity_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/high_temp_degree_days_return_level_100yr", "threshold": 35, "name": "extreme_heat_ecmwf_2014-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/FIRMS_FRP_90th_percentile", "threshold": 37.89, "name": "fire_FRP_nasa_2001-2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/FIRMS_count_90th_percentile", "threshold": 4.91, "name": "fire_frequency_nasa_2001-2023", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/sand_dust_storm_annual", "threshold": 0, "name": "sand_dust_storm_unccd_2024", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/pm25_p90_1998_2023", "threshold": 5, "name": "air_pollution_pm25_1998-2023", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/Pv_average_2013_2022", "threshold": 0.001, "name": "vectorborne_malariapv_2012-2022", "type": "Image"},
    {"id": "projects/unicef-ccri/assets/hazards/Pf_average_2013_2022", "threshold": 0.001, "name": "vectorborne_malariapf_2012-2022", "type": "Image"}
]


hazardTopics = {
    'river_flood': ['river_flood_100yr_jrc_2024'],
    'coastal_flood': ['coastal_flood_100yr_jrc_2024'],
    'storm': ['tropical_storm_100yr_giri_2024'],
    'drought': ['agricultural_drought_fao_1984-2023', 'drought_spei_copernicus_1940-2024', 'drought_spi_copernicus_1940-2024'],
    'heatwave': ['heatwave_frequency_ecmwf_2014-2024', 'heatwave_duration_ecmwf_2014-2024', 'heatwave_severity_ecmwf_2014-2024'],
    'extreme_heat': ['extreme_heat_ecmwf_2014-2024'],#version with only frequncy and extreme heat
    'fire': ['fire_FRP_nasa_2001-2024', 'fire_frequency_nasa_2001-2023'],
    'sand_dust': ['sand_dust_storm_unccd_2024']#,
    #'air_pollution': ['air_pollution_pm25_1998-2023'],
    #'malaria': ['vectorborne_malariapv_2012-2022', 'vectorborne_malariapf_2012-2022']
}


# =========================================================
# 3. DYNAMIC THRESHOLD & MASKING LOGIC
# =========================================================

# 1. Compute 'Mean' Thresholds Dynamically
# Compute Mean thresholds (keep SERVER SIDE)
# for h in hazards:
#     if h['threshold'] == 'Mean':
#         # Load correctly based on declared type
#         layer = (
#             ee.ImageCollection(h['id']).mosaic()
#             if h['type'] == 'Collection'
#             else ee.Image(h['id'])
#         )
#         th = layer.reduceRegion(
#             reducer=ee.Reducer.mean(),
#             geometry=global_geom,
#             scale=targetScale,
#             bestEffort=True,
#             maxPixels=1e13
#         ).values().get(0)
#         # Store server-side number (DO NOT .getInfo())
#         h['threshold'] = ee.Number(th)

# 2. Functional Binary Mask Generator
def get_binary_mask(h):
    layer = (
        ee.ImageCollection(h['id']).mosaic()
        if h['type'] == 'Collection'
        else ee.Image(h['id'])
    )
    th = h['threshold']
    # FAO drought index: values 0–100
    if h['name'] == "agri_drought_fao":
        return layer.lte(100).And(layer.gt(th))
    # SPI / SPEI: negative drought threshold
    if "spei" in h['name'] or "spi" in h['name']:
        layer = layer.updateMask(layer.gt(-10))
        return layer.lt(th)
    # Default hazards
    return layer.gt(th)

# 3. Generate Analysis Layers
exposureByHazardName = {h['name']: get_binary_mask(h) for h in hazards}

topicExposureImages = {}
for topic, names in hazardTopics.items():
    masks = [exposureByHazardName[name].unmask(0) for name in names]
    unionMask = masks[0]
    for m in masks[1:]:
        unionMask = unionMask.Or(m)
    topicExposureImages[topic] = unionMask

hazard_count_layer = ee.Image.cat([m.unmask(0) for m in exposureByHazardName.values()]).reduce(ee.Reducer.sum())

stacked = ee.Image.cat(list(topicExposureImages.values()))
topicCount = stacked.reduce(ee.Reducer.sum()).rename('topic_count')

# =========================================================
# 4. EXECUTION LOOP & EXPORT
# =========================================================

country_list = hybridBoundaries.aggregate_array('ucode').getInfo()

# Define the allowed codes - remove to run for all countries
allowed_codes = ['MOZ_V1']
for ucode in country_list:
    if not ucode: continue
    if ucode not in allowed_codes:
     continue
    country_feature = hybridBoundaries.filter(ee.Filter.eq('ucode', ucode)).first()
     # Get stscod
    stscod = country_feature.get('type').getInfo()

    # Skip if stscod is not 'State'
    if stscod != 'State':
        print(f"Skipping ucode {ucode} due to stscod = {stscod}")
        continue

    if ucode in ['NZL_V1', 'ATA_V1']:
      country_feature = simple_fc.filter(ee.Filter.eq('ucode', ucode)).first()
    geom = country_feature.geometry()
    iso3 = country_feature.get('ISO3')
    adm0_name = country_feature.get('name')

    # Demographic Baselines
    totalpop_sum = totalpop.reduceRegion(ee.Reducer.sum(), geom, totalpop_res, maxPixels=1e13).get('b1')
    childpop_total = childpop.reduceRegion(ee.Reducer.sum(), geom, pop_res, maxPixels=1e13).get('b1')

    results = []

    # A. Individual Hazards (with 5km Coastal Buffer)
    for name, mask in exposureByHazardName.items():
        analysis_geom = geom.buffer(5000) if "coastal" in name else geom
        val = childpop.updateMask(mask).reduceRegion(ee.Reducer.sum(), analysis_geom, pop_res, bestEffort=True, maxPixels=1e13).get('b1')
        results.append(ee.Feature(None, {'iso3': iso3, 'ucode':ucode, 'adm0_name': adm0_name, 'metric':'individual', 'label':name, 'child_population_exposed':val, 'child_population_total':childpop_total, 'population_total':totalpop_sum}))

    # B. Thematic Topics
    for n in range(1, 9):
        val = childpop.updateMask(topicCount.gte(n)).reduceRegion(ee.Reducer.sum(), geom, pop_res, bestEffort=True, maxPixels=1e13).get('b1')
        results.append(ee.Feature(None, {'iso3': iso3, 'ucode':ucode, 'adm0_name': adm0_name,'metric':'topic_count', 'label':f'ge_{n}_topic', 'child_population_exposed':val, 'child_population_total':childpop_total, 'population_total':totalpop_sum}))

    # C. Multi-Hazard Overlap (1 to 16)
    for n in range(1, 17):
        val = childpop.updateMask(hazard_count_layer.gte(n)).reduceRegion(ee.Reducer.sum(), geom, pop_res, bestEffort=True, maxPixels=1e13).get('b1')
        results.append(ee.Feature(None, {'iso3': iso3, 'ucode':ucode, 'adm0_name': adm0_name, 'metric':'multi_hazard_count', 'label':f'ge_{n}', 'child_population_exposed':val, 'child_population_total':childpop_total, 'population_total':totalpop_sum}))

    # D. Hazards Topics (with 5km Coastal Buffer)
    topics_list = ['drought','heatwave','fire','malaria']
    for name, mask in topicExposureImages.items():
        if name not in topics_list:
          continue
        val = childpop.updateMask(mask).reduceRegion(ee.Reducer.sum(), geom, pop_res, bestEffort=True, maxPixels=1e13).get('b1')
        results.append(ee.Feature(None, {'iso3': iso3, 'ucode':ucode, 'adm0_name': adm0_name, 'metric':'topic', 'label':f'{name}_topic', 'child_population_exposed':val, 'child_population_total':childpop_total, 'population_total':totalpop_sum}))

    # Export

    ee.batch.Export.table.toDrive(
        collection=ee.FeatureCollection(results),
        description=f'Exp_Analysis_{ucode}_spi_spei_-1.5_period_1945_2025',
        folder='p1_exposure_test',
        fileFormat='CSV'
    ).start()
    time.sleep(2)
    print(f"Submitted task for {ucode}")

In [ ]:
#combine output files by hazard

In [2]:
import os
import glob
import pandas as pd
output_folder = 'p1_exposure_topic_adm0_2026'

# Define the folder and get all CSV file paths
hazard_folder = os.path.join('/content/drive/MyDrive', output_folder)
hazard_files = glob.glob(f'{hazard_folder}/*.csv')

# Read and concatenate all CSVs into one DataFrame
merged_df = pd.concat([pd.read_csv(f) for f in hazard_files], ignore_index=True)

# Output directory
output_dir = '/content/drive/MyDrive/p1_exposure'
os.makedirs(output_dir, exist_ok=True)

# Loop through each unique hazard and export a CSV
for hazard_name in merged_df['label'].unique():
    # Filter the data
    df_subset = merged_df[merged_df['label'] == hazard_name]

    # Define full path
    output_path = os.path.join(output_dir, f"{hazard_name}_exposure_adm0.csv")

    # Save the CSV
    df_subset.to_csv(output_path, index=False)

    print(f"✅ Saved: {output_path}")
